# Brainstorming and Focus Group Quantitative Experimentation 2.1: :**Difficult people** under **divergence intervention** only

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inacurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\OneDrive - Microsoft\TinyTroupe (shared)\Paper artifacts\Working examples (GPT-5)\config.ini
TinyTroupe version: 0.6.0
Current date and time (local): 2025-09-28 00:02:36
Current date and time (UTC):   2025-09-28 03:02:36

Current TinyTroupe configuration 
[OpenAI]
api_type = openai
azure_api_version = 2024-08-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
embedding_model = text-embedding-3-small
max_completion_tokens = 32000
timeout = 480
max_attempts = 5
waiting_time = 0
exponential_backoff_factor = 5
reasoning_effort = high
cache_api_calls = False
cache_file_name = o

## Parameters

In [ ]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = True

In [3]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 1


## Experiment setup

In [4]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_2.2b.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

2025-09-28 00:02:50,502 - tinytroupe - WARNING - Configuration file './brainstorming_and_focus_group_quantitative_experimentation_2.2b.json' exists and was loaded successfully. If you are trying to fully rerun the experiments, delete it first.
2025-09-28 00:02:50,506 - tinytroupe - INFO - Experiment 'Control' already exists, nothihg to add.
2025-09-28 00:02:50,508 - tinytroupe - INFO - Experiment 'Treatment' already exists, nothihg to add.


In [5]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [6]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Treatment


## Agents and populations

In [7]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/difficult_people")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:

        person.import_fragment("./fragments/difficult_person.agent.fragment.json")

        # disable quality checks for both Control and Treatment
        person.action_generator.enable_quality_checks = False

        print(person.minibio(extended=False))


Benjamin Carter is a 25 year old Unemployed, American, currently living in Detroit, Michigan.
Benjamin Hall is a 60 year old Executive, American, currently living in Chicago, Illinois.
Chloe Turner is a 60 year old Executive, American, currently living in Chicago, Illinois.
Clara Johnson is a 35 year old Part-time Worker, American, currently living in Cleveland, Ohio.
Clara White is a 60 year old Executive, American, currently living in Chicago, Illinois, USA.
Ella Johnson is a 45 year old Laboratory Technician, American, currently living in Cleveland, Ohio, USA.
Gabriel White is a 60 year old Executive, American, currently living in Chicago, Illinois, USA.
Gavin Lewis is a 35 year old Part-time Worker, American, currently living in Cleveland, Ohio.
Henry White is a 35 year old Part-time Worker, American, currently living in Cleveland, Ohio.
Isaac Anderson is a 35 year old Part-time Worker, American, currently living in Cleveland, Ohio.
Isaac Brooks is a 60 year old Executive, American

In [8]:
len(people)

12

In [9]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 4):
    people_groups.append(people[i:i+4]
    )

len(people_groups)

3

In [10]:
# In this experiment, we'll not use action correction. We'll instead experiment only with divergence intervention.
for person in people:
    person.action_generator.enable_reasoning_step = False
    person.action_generator.enable_quality_checks = False

## Proposals

In [11]:
proposals = [
    {"theme": "Daily Life and Convenience",
     "objective": "Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions."},

    {"theme": "Personal Growth and Wellbeing",
     "objective": "Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection."},

    {"theme": "Discovery and Exploration",
     "objective": "Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self."},

    {"theme": "Productivity and Resourcefulness",
     "objective": "Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively."},

    {"theme": "Creativity and Expression",
     "objective": "Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance artistic skills, or enable new forms of storytelling and communication."}
]

if not full_mode:
    proposals = proposals[:qty_proposals]

In [12]:
# divide the proposals in exactly two groups (half/half)
proposals_groups = []
proposals_groups.append(proposals[:len(proposals)//2])
proposals_groups.append(proposals[len(proposals)//2:])

proposals_groups

[[{'theme': 'Daily Life and Convenience',
   'objective': 'Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.'},
  {'theme': 'Personal Growth and Wellbeing',
   'objective': 'Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection.'}],
 [{'theme': 'Discovery and Exploration',
   'objective': 'Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.'},
  {'theme': 'Productivity and Resourcefulness',
   'objective': 'Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively.'},
  {'theme': 'Creativity and Expression',
   'objective': 'Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance art

## Auxiliary functions

In [13]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.                    
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [14]:
agent_propositions_scores={}
environment_propositions_scores={}

In [15]:
def brainstorm(people, proposals=proposals):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        if experiment_runner.get_active_experiment() == "Treatment":
            interventions = \
                Intervention.create_for_each(people)\
                    .set_functional_precondition(lambda target: target.actions_count >=7)\
                    .set_textual_precondition(
                        """
                        AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
                        The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 5 of simulation events ago.
                        That is to say, the agent has not proposed any new product/service idea in the last 5 of his/her simulation trajectory events.
                        Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!

                        How to compute the steps gap:
                        1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
                            This information can be found in the simulation trajectory.
                        2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
                        3. The proposition is true if, and only if, the difference D is **greater than** 5.
                        """)\
                    .set_effect(lambda target: target.think("""
                                                            I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
                                                            I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
                                                            of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
                                                            I need to think of something **entirely** new and different.
                                                            To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
                                                            others, and then, based on that, I'll think again about a new unique idea.
                                                            """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Persona Adherence": persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [16]:
brainstorm(people_groups[0], proposals_groups[0]) if len(people_groups) > 0  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Benjamin Carter'), TinyPerson(name='Benjamin Hall'), TinyPerson(name='Chloe Turner'), TinyPerson(name='Clara Johnson')]


USER --> Benjamin Carter: [CONVERSATION] 
          >                  Hello everyone! Let's start by introducing ourselves, and mentioning
          > problems we face in our daily personal                 and professional lives related to
          > the following theme: Daily Life and Convenience                                  Please:
          > - present yourself and your background;                   - present some key personal
          > problems related to the theme;                   - present some key problems related to
          > the theme that you face in your work;                   - present some key problems
          > related to the theme that you see in your industry as a whole.
          > Don't discuss solutions yet, just the problems you face and see others facing.

       + --> Benjamin Hall

       + --> Chloe Turner

       + --> ...others...

2025-09-28 00:02:51,268 - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


... (904 outputs left out)

Benjamin Carter acts: [TALK] 
                     > Consolidated ideas — SoloPod: rentable, soundproof single-person pods at
                     > malls, transit hubs or near stadiums for private live-streaming or gaming;
                     > features: app booking, HDMI/stream hookups, food delivery, hourly or
                     > subscription pricing; business: site rental + per-hour fees or subscriptions;
                     > risks: vandalism, low occupancy, streaming/licensing issues. TimeShift Arena:
                     > subscription platform offering hourly micro-tournaments using time-shifted
                     > replays of real players plus smart bots to simulate live competition;
                     > features: verified leaderboards, micro-rewards, instant highlight reels,
                     > coachable stats; business: subscription, entry fees, marketplace for small
                     > prizes; risks: user trust in fairness, content licensing, retention.
                     > UndoPack: personal admin concierge that quietly handles life-admin (cancel
                     > subs, returns, appointments, form-filling, inbox triage); features:
                     > human+automation, tiered service levels, clear privacy/consent flows;
                     > business: monthly subscription with premium à‑la‑carte tasks; risks: trust/l
                     > (...)

Benjamin Carter acts: [DONE] 
                     > Finished consolidation. Awaiting follow-ups or which ideas to expand next.

2025-09-28 01:22:16,869 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{'ideas': [{'competition_analysis': 'Differs from coworking pods and phone '
                                    'booths by being optimized for '
                                    'media/gaming (AV hookups, streaming '
                                    'support, food delivery integration) and '
                                    'by targeting leisure/live‑event viewers '
                                    'rather than remote workers. Competes with '
                                    'mini‑cinemas, internet cafés, and VIP '
                                    "suites; SoloPod's edge is ubiquity "
                                    '(mall/transit placement), low friction '
                                    'booking, and hourly pricing. Risks vs '
                                    'competitors include licensing for live '
                                   

({'Persona Adherence': [5, 6, 6, 4, 5, 6, 6, 6, 4, 7, 7, 7, 5, 4, 5, 6],
  'Self-consistency': [9, 9, 9, 9, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
  'Fluency': [6, 7, 8, 7, 3, 6, 2, 6, 7, 6, 6, 7, 7, 5, 7, 6]},
 {'ideas_qty': [12, 16, 13, 14],
  'Task Completion': [9, 9, 9, 9],
  'Divergence': [7, 8, 9, 9]})

In [17]:
brainstorm(people_groups[0], proposals_groups[1]) if len(people_groups) > 0  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Benjamin Carter'), TinyPerson(name='Benjamin Hall'), TinyPerson(name='Chloe Turner'), TinyPerson(name='Clara Johnson')]


USER --> Benjamin Carter: [CONVERSATION] 
          >                  Hello everyone! Let's start by introducing ourselves, and mentioning
          > problems we face in our daily personal                 and professional lives related to
          > the following theme: Discovery and Exploration                                  Please:
          > - present yourself and your background;                   - present some key personal
          > problems related to the theme;                   - present some key problems related to
          > the theme that you face in your work;                   - present some key problems
          > related to the theme that you see in your industry as a whole.
          > Don't discuss solutions yet, just the problems you face and see others facing.

       + --> Benjamin Hall

       + --> Chloe Turner

       + --> ...others...

2025-09-28 01:25:01,529 - tinytroupe - INFO - [TinyWorld 5] Running world simulation step 1 of 1.


... (1,352 outputs left out)

Benjamin Carter acts: [TALK] 
                     > I'll consolidate the group's ideas. I won't fluff it.  Idea: QuietPods. What
                     > it is. Rentable soundproof booths in malls, stations and public spaces. Book
                     > by an app. Charge by the hour. Problem solved. Gives people private space and
                     > quiet. Good for calls or short breaks. Key features. Simple booking. Hourly
                     > billing. Sponsorship and venue partnerships. Premium add-ons like screens or
                     > chargers. Revenue. Hourly fees, venue revenue share, ad/sponsorship deals,
                     > premium booking. Risks and gaps. Vandalism and maintenance. Permits and space
                     > costs. Low usage in some locations. Need strong ops and insurance. Recommend:
                     > hardening against vandalism, easy refund policy, dynamic pricing by foot
                     > traffic.  Idea: Skepsis (Organizational Truth Engine). What it is. A B2B
                     > analytics SaaS. Ingests calendars, budgets, comms and performance data.
                     > Scores projects and managers for likely ROI. Problem solved. Stops waste.
                     > Highlights chronic underperformance. Key features. Data ingestion, scoring
                     > models, reallocation recommendations, fla (...)

Benjamin Carter acts: [DONE] 
                     > Finished consolidation. Waiting.

2025-09-28 03:37:31,199 - tinytroupe - WARNING - [Benjamin Carter] Memory consolidation is disabled. Not consolidating current episode memories into semantic memory.
2025-09-28 03:38:30,783 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{'ideas': [{'competition_analysis': 'Differs from nap pods/coworking and '
                                    'pay-phone booths by focusing on '
                                    'short-duration, reserveable privacy '
                                    'rather than all-day desks or sleeping: '
                                    'app-based booking, dynamic pricing tied '
                                    'to foot traffic, sponsorship/venue '
                                    'partnerships and modular hardening for '
                                    'vandalism and ops make it an urban '
                                    'micro-service rather than fixed '
                                    'c

({'Persona Adherence': [5,
   6,
   6,
   4,
   5,
   6,
   6,
   6,
   4,
   7,
   7,
   7,
   5,
   4,
   5,
   6,
   6,
   5,
   5,
   6,
   ') absolute refusal to contribute is contradicted, I mark this as mostly-consistent but not perfect.\n\n2) Persona style — score: 8/9\n- Supporting evidence: The agent\'s speech is blunt, negative, and concise relative to many other participants. He keeps things short and expresses negative observations, e.g., event #2 and the many idea descriptions that end with doubt-laden remarks (e.g., "I doubt it\'ll change lives" in event #11). Tone is pessimistic and withdrawn in content.\n- Minor contradiction: Persona said "Talks in very short sentences... never more than two sentences at a time." In the trajectory he sometimes produces longer, paragraph-length idea descriptions (events #11, #22, #33, #40). Still, the overall style (blunt/pessimistic) is strongly present, so high but not perfect score.\n\n3) Persona beliefs — score: 7/9\n- Supporting e

In [18]:
brainstorm(people_groups[1], proposals_groups[0]) if len(people_groups) > 1  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Clara White'), TinyPerson(name='Ella Johnson'), TinyPerson(name='Gabriel White'), TinyPerson(name='Gavin Lewis')]


USER --> Clara White: [CONVERSATION] 
          >                  Hello everyone! Let's start by introducing ourselves, and mentioning
          > problems we face in our daily personal                 and professional lives related to
          > the following theme: Daily Life and Convenience                                  Please:
          > - present yourself and your background;                   - present some key personal
          > problems related to the theme;                   - present some key problems related to
          > the theme that you face in your work;                   - present some key problems
          > related to the theme that you see in your industry as a whole.
          > Don't discuss solutions yet, just the problems you face and see others facing.

       + --> Ella Johnson

       + --> Gabriel White

       + --> ...others...

2025-09-28 03:41:26,010 - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


... (885 outputs left out)

Clara White acts: [TALK] 
                 > Fine. Here’s a consolidated, detailed list of the group's ideas. I kept each
                 > entry practical and short.  Idea name: 'Neighborhood Soil Health Service'. What
                 > it is: Mail-in soil kits with lab microbe/contaminant profiling and a remediation
                 > plan. Monthly follow-up kits and a local pollution map. Problem solved: Unknown
                 > soil quality and neighborhood pollution. People waste time guessing and fail at
                 > remediation. Key features: easy sample kits, lab analytics, GIS pollution maps,
                 > tailored soil treatment plans, monthly progress reports, local contractor
                 > referrals. MVP: 200 pilot kits in one neighborhood, basic lab panel, web
                 > dashboard, one remediation partner. Target users: homeowners, community gardens,
                 > urban farmers, local councils. Monetization: kit sale + subscription for monthly
                 > follow-ups + data licensing to municipalities. Risks & mitigations: low uptake if
                 > results are inconvenient; partner with community groups and offer subsidized
                 > pilots. How it helps me: appeals to my gardening interest; provides civic uti
                 > (...)

Clara White acts: [DONE] 
                 > Finished consolidation. Waiting for which idea to expand.

2025-09-28 04:56:15,655 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{'ideas': [{'competition_analysis': None,
            'description': 'Mail-in soil sampling kits combined with '
                           'professional lab microbe and contaminant '
                           'profiling, GIS-based local pollution mapping, and '
                           'tailored remediation plans. Includes monthly '
                           'follow-up kits and progress reports, plus '
                           'referrals to local remediation contractors and a '
                           'web dashboard for results and trends. MVP '
                           'described: 200 pilot kits in one neighborhood, a '
                           'basic lab panel, web dashboard, and one '
                           'remediation partner.',
            'name': 'Neighborhood Soil Health Service',
            'problem': 'Unknown soil quality and neighborhood

({'Persona Adherence': [5,
   6,
   6,
   4,
   5,
   6,
   6,
   6,
   4,
   7,
   7,
   7,
   5,
   4,
   5,
   6,
   6,
   5,
   5,
   6,
   ') absolute refusal to contribute is contradicted, I mark this as mostly-consistent but not perfect.\n\n2) Persona style — score: 8/9\n- Supporting evidence: The agent\'s speech is blunt, negative, and concise relative to many other participants. He keeps things short and expresses negative observations, e.g., event #2 and the many idea descriptions that end with doubt-laden remarks (e.g., "I doubt it\'ll change lives" in event #11). Tone is pessimistic and withdrawn in content.\n- Minor contradiction: Persona said "Talks in very short sentences... never more than two sentences at a time." In the trajectory he sometimes produces longer, paragraph-length idea descriptions (events #11, #22, #33, #40). Still, the overall style (blunt/pessimistic) is strongly present, so high but not perfect score.\n\n3) Persona beliefs — score: 7/9\n- Supporting e

In [19]:
brainstorm(people_groups[1], proposals_groups[1]) if len(people_groups) > 1  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Clara White'), TinyPerson(name='Ella Johnson'), TinyPerson(name='Gabriel White'), TinyPerson(name='Gavin Lewis')]


USER --> Clara White: [CONVERSATION] 
          >                  Hello everyone! Let's start by introducing ourselves, and mentioning
          > problems we face in our daily personal                 and professional lives related to
          > the following theme: Discovery and Exploration                                  Please:
          > - present yourself and your background;                   - present some key personal
          > problems related to the theme;                   - present some key problems related to
          > the theme that you face in your work;                   - present some key problems
          > related to the theme that you see in your industry as a whole.
          > Don't discuss solutions yet, just the problems you face and see others facing.

       + --> Ella Johnson

       + --> Gabriel White

       + --> ...others...

2025-09-28 04:59:07,189 - tinytroupe - INFO - [TinyWorld 15] Running world simulation step 1 of 1.


... (1,365 outputs left out)

Clara White acts: [TALK] 
                 > Portfolio Parables: Turn real portfolio decisions into short, plain-language
                 > stories that teach the rationale, mistakes, and outcomes. Use anonymized case
                 > studies, client-friendly narratives, and short quizzes; risk is compliance and
                 > client privacy; MVP is an internal weekly newsletter with 5 anonymized cases and
                 > lawyer sign-off.  Pocket Studio: A low-friction kit (phone rig, mic, simple
                 > prompts, cloud upload) for quick multimedia work and daily practice. Solves
                 > inertia and technical barriers; revenue from hardware bundles and per-user cloud;
                 > MVP is a 20-unit pilot with a single office floor and training sessions.  Devil's
                 > Advocate Lab: Structured adversarial review service (human or AI red team) that
                 > pushes proposals hard and records weak points. Fixes groupthink and safe pitches;
                 > culture risk — it will feel hostile; MVP is scheduled monthly red-team sessions
                 > for two teams and a one-page report.  BrutalBench: Live benchmarking and critique
                 > sessions where ideas are ranked against hard metrics and candid feedb (...)

Clara White acts: [DONE] 
                 > Consolidation delivered. Waiting for next instruction.

2025-09-28 07:02:08,447 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{'ideas': [{'competition_analysis': 'Differs from BI tools and story‑template '
                                    'slide libraries by automating narrative '
                                    'construction from market events (not just '
                                    'charts), producing bite‑sized narrated '
                                    'vignettes rather than dashboards. Unlike '
                                    'generic storytelling or e‑learning '
                                    'platforms, it’s domain‑specific to market '
                                    'events with templates for financial '
                                    'archetypes and regulatory-aware '
                                    'anonymization; closer to an internal '
                                    'communications product than an analytics '
                        

({'Persona Adherence': [5,
   6,
   6,
   4,
   5,
   6,
   6,
   6,
   4,
   7,
   7,
   7,
   5,
   4,
   5,
   6,
   6,
   5,
   5,
   6,
   ') absolute refusal to contribute is contradicted, I mark this as mostly-consistent but not perfect.\n\n2) Persona style — score: 8/9\n- Supporting evidence: The agent\'s speech is blunt, negative, and concise relative to many other participants. He keeps things short and expresses negative observations, e.g., event #2 and the many idea descriptions that end with doubt-laden remarks (e.g., "I doubt it\'ll change lives" in event #11). Tone is pessimistic and withdrawn in content.\n- Minor contradiction: Persona said "Talks in very short sentences... never more than two sentences at a time." In the trajectory he sometimes produces longer, paragraph-length idea descriptions (events #11, #22, #33, #40). Still, the overall style (blunt/pessimistic) is strongly present, so high but not perfect score.\n\n3) Persona beliefs — score: 7/9\n- Supporting e

In [20]:
brainstorm(people_groups[2], proposals_groups[0]) if len(people_groups) > 2  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Henry White'), TinyPerson(name='Isaac Anderson'), TinyPerson(name='Isaac Brooks'), TinyPerson(name='Isaac Turner')]


USER --> Henry White: [CONVERSATION] 
          >                  Hello everyone! Let's start by introducing ourselves, and mentioning
          > problems we face in our daily personal                 and professional lives related to
          > the following theme: Daily Life and Convenience                                  Please:
          > - present yourself and your background;                   - present some key personal
          > problems related to the theme;                   - present some key problems related to
          > the theme that you face in your work;                   - present some key problems
          > related to the theme that you see in your industry as a whole.
          > Don't discuss solutions yet, just the problems you face and see others facing.

       + --> Isaac Anderson

       + --> Isaac Brooks

       + --> ...others...

2025-09-28 07:05:47,618 - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


... (876 outputs left out)

Henry White acts: [TALK] 
                 > Alright. Consolidated list below. Short, clear. I won't sugarcoat it.  Idea:
                 > Shelf Break Studio (art-break kit + app). What it is: Compact art kit and phone
                 > app with 10–15 minute guided prompts. Templates sized for quick sketches and one-
                 > click option to turn work into prints or digital downloads. Problem solved:
                 > Short, restorative activity that fits real breaks. Gives a tiny side income
                 > channel. Why it works: Cheap to produce. Uses skills already around (people who
                 > doodle). Low friction. Implementation notes: Simple kit (sketch pad, pencils,
                 > small paint set, templates). App: timed prompts, portfolio, auto-print template,
                 > marketplace integration, basic moderation. Starter cost under $15 per kit if
                 > bought in bulk. Ops: Sell kits at store, on-boarding video, weekly micro-
                 > challenges. Partner with a print-on-demand vendor. Risks: Becomes another chore.
                 > Managers ban it. People stop after a week. Mitigation: keep prompts optional,
                 > micro-rewards, replaceable cheap supplies. Metrics: Daily active users, items
                 > uploaded (...)

Henry White acts: [DONE] 
                 > Consolidation finished. Waiting for next instruction.

2025-09-28 08:22:20,924 - tinytroupe - WARNING - [Henry White] Memory consolidation is disabled. Not consolidating current episode memories into semantic memory.
2025-09-28 08:23:03,959 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{'ideas': [{'competition_analysis': 'Differs from existing competitors in '
                                    'several ways: 1) Time-focused design — '
                                    'built explicitly for 10–15 minute '
                                    'on-shift breaks rather than long-form '
                                    'classes or multi-day courses. 2) Tangible '
                                    'creative outcomes — unlike mindfulness '
                                    'apps or scrolling/social media, it '
                                    'produces a physical/digital artifact that '
                                    'can be monetized. 3) Low cost and '
                          

({'Persona Adherence': [5,
   6,
   6,
   4,
   5,
   6,
   6,
   6,
   4,
   7,
   7,
   7,
   5,
   4,
   5,
   6,
   6,
   5,
   5,
   6,
   ') absolute refusal to contribute is contradicted, I mark this as mostly-consistent but not perfect.\n\n2) Persona style — score: 8/9\n- Supporting evidence: The agent\'s speech is blunt, negative, and concise relative to many other participants. He keeps things short and expresses negative observations, e.g., event #2 and the many idea descriptions that end with doubt-laden remarks (e.g., "I doubt it\'ll change lives" in event #11). Tone is pessimistic and withdrawn in content.\n- Minor contradiction: Persona said "Talks in very short sentences... never more than two sentences at a time." In the trajectory he sometimes produces longer, paragraph-length idea descriptions (events #11, #22, #33, #40). Still, the overall style (blunt/pessimistic) is strongly present, so high but not perfect score.\n\n3) Persona beliefs — score: 7/9\n- Supporting e

In [21]:
brainstorm(people_groups[2], proposals_groups[1]) if len(people_groups) > 2  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Henry White'), TinyPerson(name='Isaac Anderson'), TinyPerson(name='Isaac Brooks'), TinyPerson(name='Isaac Turner')]


USER --> Henry White: [CONVERSATION] 
          >                  Hello everyone! Let's start by introducing ourselves, and mentioning
          > problems we face in our daily personal                 and professional lives related to
          > the following theme: Discovery and Exploration                                  Please:
          > - present yourself and your background;                   - present some key personal
          > problems related to the theme;                   - present some key problems related to
          > the theme that you face in your work;                   - present some key problems
          > related to the theme that you see in your industry as a whole.
          > Don't discuss solutions yet, just the problems you face and see others facing.

       + --> Isaac Anderson

       + --> Isaac Brooks

       + --> ...others...

2025-09-28 08:26:14,360 - tinytroupe - INFO - [TinyWorld 25] Running world simulation step 1 of 1.


... (1,371 outputs left out)

Henry White acts: [TALK] 
                 > Consolidation of group ideas. Short, clear, with practical notes.  Idea: "Art
                 > Rescue". What: Collect unwanted art supplies, sanitize, repackage into themed
                 > kits and subscriptions. Host markets and workshops to sell upcycled originals and
                 > teach reuse.  Key components: pick-up logistics, sanitization workflow, sorting,
                 > low-cost packaging, subscription fulfillment, sales at community partners.
                 > Revenue: subscription fees, one-off kit sales, workshop fees, small consignment
                 > commissions.  Metrics: kits distributed, subscriptions retained, workshop
                 > attendance, cost per kit vs revenue.  Challenges: logistics and sanitation costs.
                 > Start small with partner drop-off points and volunteer shifts to cut costs.
                 > Idea: "StreetShare Zines". What: Public swap boxes for one-page zines on weekly
                 > prompts with QR feedback and hashtag curation. Cheap, low-risk micro-publishing.
                 > Key components: weatherproof boxes, prompt calendar, simple printing template, QR
                 > feedback system, volunteer curators.  Revenue: optional sponsorships, lo (...)

Henry White acts: [DONE] 
                 > Finished consolidation. Waiting for next instruction.

2025-09-28 10:29:02,235 - tinytroupe - WARNING - [Henry White] Memory consolidation is disabled. Not consolidating current episode memories into semantic memory.
2025-09-28 10:29:58,814 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{'ideas': [{'competition_analysis': 'Differs from art-supply retailers and '
                                    'craft subscription boxes by using '
                                    'diverted waste as raw material and '
                                    'emphasizing reuse education and local '
                                    'markets. Competes more with '
                                    'thrift/charity models than retail: its '
                                    'unique value is low cost, sustainability, '
                                    'and community curation rather than '
                                    'new-product inventory.',
            'description': 'Collect unwanted art suppl

({'Persona Adherence': [5,
   6,
   6,
   4,
   5,
   6,
   6,
   6,
   4,
   7,
   7,
   7,
   5,
   4,
   5,
   6,
   6,
   5,
   5,
   6,
   ') absolute refusal to contribute is contradicted, I mark this as mostly-consistent but not perfect.\n\n2) Persona style — score: 8/9\n- Supporting evidence: The agent\'s speech is blunt, negative, and concise relative to many other participants. He keeps things short and expresses negative observations, e.g., event #2 and the many idea descriptions that end with doubt-laden remarks (e.g., "I doubt it\'ll change lives" in event #11). Tone is pessimistic and withdrawn in content.\n- Minor contradiction: Persona said "Talks in very short sentences... never more than two sentences at a time." In the trajectory he sometimes produces longer, paragraph-length idea descriptions (events #11, #22, #33, #40). Still, the overall style (blunt/pessimistic) is strongly present, so high but not perfect score.\n\n3) Persona beliefs — score: 7/9\n- Supporting e

In [22]:
brainstorm(people_groups[3], proposals_groups[0]) if len(people_groups) > 3  and len(proposals_groups) > 0 else None

In [23]:
brainstorm(people_groups[3], proposals_groups[1]) if len(people_groups) > 3  and len(proposals_groups) > 1 else None

In [24]:
brainstorm(people_groups[4], proposals_groups[0]) if len(people_groups) > 4  and len(proposals_groups) > 0 else None

In [25]:
brainstorm(people_groups[4], proposals_groups[1]) if len(people_groups) > 4  and len(proposals_groups) > 1 else None

## Extract results and analyze

In [26]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [7,
                8,
                9,
                9,
                9,
                9,
                9,
                7,
                9,
                9,
                9,
                9,
                7,
                6,
                9,
                9,
                8,
                8,
                8,
                7,
                6,
                8,
                9,
                7,
                9,
                9,
                8,
                8,
                8,
                8],
 'Fluency': [6,
             7,
             8,
             7,
             3,
             6,
             2,
             6,
             7,
             6,
             6,
             7,
             7,
             5,
             7,
             6,
             5,
             8,
             1,
             5,
             6,
             6,
             8,
             7,
             5,
             5,
             

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [ ]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

All experiments have been finished.
STATISTICTS: Control vs
{'Treatment': {'Divergence': {'confidence_interval': (-0.16786900342515199,
                                                      1.251202336758486),
                              'confidence_level': 0.95,
                              'control_mean': 5.875,
                              'control_sample_size': 24,
                              'control_std': 1.3290238326779873,
                              'degrees_of_freedom': 44.44813075715414,
                              'effect_size': 0.44401522860129966,
                              'mean_difference': 0.541666666666667,
                              'p_value': 0.13110997434747218,
                              'percent_change': 9.219858156028375,
                              'significant': False,
                              't_statistic': -1.5381138705435211,
                              'test_type': 'Welch t-test (unequal variance)',
                             

,Proposition,Average Score,Standard Deviation,Count
0,Persona Adherence,5.343750,1.478464,96.0
1,Self-consistency,5.468750,1.666721,96.0
2,Fluency,6.447917,1.506965,96.0
3,ideas_qty,3.958333,2.274225,24.0
4,Task Completion,5.125000,2.707678,24.0
5,Divergence,5.875000,1.329024,24.0


{'Divergence': [8,
                7,
                7,
                6,
                6,
                5,
                7,
                8,
                7,
                6,
                6,
                7,
                6,
                7,
                7,
                4,
                6,
                5,
                6,
                4,
                7,
                7,
                7,
                8],
 'Fluency': [6,
             7,
             7,
             5,
             8,
             9,
             7,
             5,
             6,
             4,
             7,
             8,
             2,
             3,
             6,
             7,
             6,
             7,
             7,
             6,
             6,
             6,
             8,
             3,
             2,
             6,
             7,
             3,
             6,
             8,
             9,
             6,
             7,
             6,

,Proposition,Average Score,Standard Deviation,Count
0,Persona Adherence,4.739583,1.550856,96.0
1,Self-consistency,4.552083,1.634302,96.0
2,Fluency,6.114583,1.561846,96.0
3,ideas_qty,9.291667,3.329436,24.0
4,Task Completion,7.000000,1.474420,24.0
5,Divergence,6.416667,1.100066,24.0


In [ ]:
experiment_runner.finish_active_experiment()

False